In [0]:
dbutils.fs.ls("abfss://source@mantristorageaccount.dfs.core.windows.net/")

In [0]:
Claims_df = spark.read.format("csv").load("abfss://source@mantristorageaccount.dfs.core.windows.net/claims/")
display(Claims_df)

In [0]:
import pyspark.sql.functions as F

data =[
     ("C001", "P001", "Submitted", 1000.00, "2026-07-20 10:00:00"),
    ("C001", "P001", "Approved",  1000.00, "2026-07-21 12:00:00"),
    ("C001", "P001", "Paid",      1000.00, "2026-07-22 15:00:00"),

    ("C002", "P002", "Submitted", 2500.00, "2026-07-20 09:00:00"),
    ("C002", "P002", "Rejected",  2500.00, "2026-07-23 11:00:00"),

    ("C003", "P003", "Submitted", 1500.00, "2026-07-21 08:00:00"),
    ("C003", "P003", "Under Review", 1500.00, "2026-07-24 14:00:00"),
    ("C003", "P003", "Under Review", 2000.00, "2026-07-25 14:00:00"),

    ("C004", "P004", "Submitted", 500.00, "2026-07-22 10:00:00"),
]

columns  = [
    "claim_id",
    "patient_id",
    "status",
    "claim_amount",
    "updated_timestamp"
]

df = spark.createDataFrame(data, schema=columns)
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#window = Window.partitionBy("patient_id").orderBy(F.col("updated_timestamp").desc())
df.createOrReplaceTempView("ClaimPaid")
df_result = spark.sql("""
               SELECT * FROM 
               ( 
                SELECT *, row_number() OVER 
                    (
                        PARTITION BY patient_id  order by updated_timestamp  DESC
                    ) as rn 
               FROM ClaimPaid where status ='Under Review'
               ) 
               WHERE rn = 1
               """)
display(df_result)
#df = df.withColumn("rank", F.row_number().over(window))

#df = df.filter(df.rank == 1).drop("rank")
#display(df)


In [0]:
df_result.printSchema()
spark.catalog.dropTempView("Claims_df")
newdf = spark.sql("""
SELECT * FROM ClaimPaid""")
display(newdf)

In [0]:
%sql
SELECT * FROM 
(
    SELECT *, row_number() over
    (
        partition by providerid order by ClaimDate desc
    ) as rank 
    FROM healthcare.bronze.claims

) WHERE rank = 1
--LIMIT 1


In [0]:
%sql
SELECT *
FROM healthcare.bronze.claims
WHERE 
   --ClaimDate >= current_date() - INTERVAL 2 DAYS; AND
   ClaimDate < current_date() + INTERVAL 1 DAY;

In [0]:
%sql
SELECT MAX(ClaimDate) FROM healthcare.bronze.claims

In [0]:
%sql
SELECT *
FROM healthcare.bronze.claims
WHERE ClaimDate = (SELECT MAX(ClaimDate) FROM healthcare.bronze.claims)
-- SELECT *
-- FROM healthcare.bronze.claims
-- WHERE ClaimDate >= (
--     SELECT add_months(MAX(ClaimDate), -6)
--     FROM healthcare.bronze.claims
-- )
-- AND ClaimDate <= (
--     SELECT MAX(ClaimDate)
--     FROM healthcare.bronze.claims
-- );